# Bankruptcy Prediction for Indian Firms
## Notebook 1: Exploratory Data Analysis & Preprocessing

**Author:** Aayush Solanki  
**Institution:** Department of Finance and Business Economics, Delhi University  
**Supervisor:** Dr. Sonal Thukral

---

### Overview

This notebook covers:
1. Loading and inspecting the dataset
2. Class distribution analysis
3. Feature-level EDA (distributions, outliers)
4. Correlation analysis and multicollinearity removal
5. Train/test split and feature scaling

**Dataset:** 525 Indian firms (75 bankrupt, 450 non-bankrupt) × 63 financial, market, and macroeconomic features.  
Data spans 2002–2022 and covers 3 years prior to each firm's last published financial report.

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from utils import set_plot_style, class_distribution
from data_preprocessing import load_dataset, remove_correlated_features, preprocess

set_plot_style()
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.4f}'.format)

## 2. Load Data

In [ ]:
DATA_PATH = '../data/raw/financial_data.xlsx'

df = load_dataset(DATA_PATH)
print(f"Shape: {df.shape}")
df.head(3)

In [ ]:
# Data types and missing values
print("Missing values per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("\nNo missing values detected — dataset is complete." if df.isnull().sum().sum() == 0 else "")

## 3. Class Distribution

The dataset is imbalanced: ~14% bankrupt vs ~86% non-bankrupt. This reflects the real-world rarity of bankruptcy events and must be considered when evaluating model performance. We rely on F1-score and ROC-AUC rather than raw accuracy as primary metrics.

In [ ]:
class_distribution(df['Bankrupt'])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Bar chart
counts = df['Bankrupt'].value_counts()
axes[0].bar(['Non-Bankrupt (0)', 'Bankrupt (1)'], counts.values, color=['#2e75b6', '#ed7d31'])
axes[0].set_ylabel('Number of Firms')
axes[0].set_title('Class Counts')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 3, str(v), ha='center', fontweight='bold')

# Pie chart
proportions = df['Bankrupt'].value_counts(normalize=True)
axes[1].pie(
    proportions.values,
    labels=['Non-Bankrupt', 'Bankrupt'],
    autopct='%1.1f%%',
    colors=['#2e75b6', '#ed7d31'],
    explode=(0, 0.08),
    startangle=90,
)
axes[1].set_title('Class Proportions')

plt.suptitle('Target Variable Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Feature Categories

The 63 predictive features are organised into six categories:

| Category | # Features | Examples |
|---|---|---|
| Profitability | 8 | ROA (EBITDA), Gross Profit Margin |
| Liquidity | 7 | Current Ratio, Quick Ratio, DSCR |
| Solvency | 3 | Debt/Equity, Total Debt/Net Worth |
| Efficiency | 7 | Fixed Assets Turnover, AR Turnover |
| Cash Flow | 7 | CFO to Assets, Cash Flow to Revenue |
| Risk / Market | 12 | EPS, ROE, Market Return, Volatility |
| Macroeconomic | 6 | GDP Growth, Inflation, FDI as % GDP |
| Financial Flags | 5 | Net Income Flag, Liability-Assets Flag |

In [ ]:
# Quick summary statistics split by bankruptcy status
numeric_df = df.select_dtypes(include='number').drop(columns=['Year'], errors='ignore')
summary = numeric_df.groupby(df['Bankrupt']).mean()
summary.index = ['Non-Bankrupt', 'Bankrupt']

# Show a few key profitability and liquidity ratios
key_cols = ['ROA (EBITDA)', 'Gross Profit Margin', 'Current Ratio', 'Quick Ratio',
            'Debt/Equity Ratio', 'CFO to Assets']
key_cols = [c for c in key_cols if c in summary.columns]
summary[key_cols].T.style.format('{:.4f}').background_gradient(cmap='RdYlGn', axis=1)

## 5. Correlation Analysis

Many financial ratios are constructed from the same underlying balance sheet items, creating multicollinearity. We remove features that are mutually correlated above |0.5| to reduce redundancy before fitting the logistic regression baseline.

In [ ]:
# Full correlation heatmap (63 features)
features_only = df.drop(columns=['Bankrupt', 'Year'], errors='ignore').select_dtypes('number')
corr = features_only.corr()

fig, ax = plt.subplots(figsize=(22, 18))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, cmap='coolwarm', center=0,
    annot=False, linewidths=0.3, ax=ax,
    cbar_kws={'shrink': 0.6},
)
ax.set_title('Correlation Heatmap — All 63 Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/correlation_heatmap_full.png', dpi=130, bbox_inches='tight')
plt.show()
print("Note: upper triangle masked for readability.")

In [ ]:
# Remove correlated features → 28-feature reduced set
df_reduced = remove_correlated_features(df)
reduced_features = df_reduced.drop(columns=['Bankrupt', 'Year'], errors='ignore').columns.tolist()
print(f"Retained features ({len(reduced_features)}):")
for f in reduced_features:
    print(f"  - {f}")

In [ ]:
# Heatmap for the 28-feature reduced set
reduced_only = df_reduced.drop(columns=['Bankrupt', 'Year'], errors='ignore').select_dtypes('number')
corr_reduced = reduced_only.corr()

fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(
    corr_reduced, cmap='coolwarm', center=0,
    annot=True, fmt='.2f', linewidths=0.4,
    annot_kws={'size': 6}, ax=ax,
    cbar_kws={'shrink': 0.7},
)
ax.set_title('Correlation Heatmap — 28 Selected Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/correlation_heatmap_selected.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Train / Test Split & Scaling

- **Split:** 70% train / 30% test (stratified on the target to preserve class ratios).
- **Scaling:** `StandardScaler` fitted on training data only — test set is transformed but never used to fit.

In [ ]:
# Preprocessing for logistic regression (28 features)
X_train_lr, X_test_lr, y_train, y_test, scaler_lr, features_lr = preprocess(
    df, apply_feature_selection=True
)

# Preprocessing for ML models (all 63 features)
X_train_ml, X_test_ml, _, _, scaler_ml, features_ml = preprocess(
    df, apply_feature_selection=False
)

print(f"\nLogistic regression feature set: {len(features_lr)} features")
print(f"ML models feature set:           {len(features_ml)} features")

In [ ]:
# Save processed data for use in subsequent notebooks
import os, pickle
os.makedirs('../data/processed', exist_ok=True)

for name, obj in [
    ('X_train_lr', X_train_lr), ('X_test_lr', X_test_lr),
    ('X_train_ml', X_train_ml), ('X_test_ml', X_test_ml),
    ('y_train', y_train), ('y_test', y_test),
    ('scaler_lr', scaler_lr), ('scaler_ml', scaler_ml),
    ('features_lr', features_lr), ('features_ml', features_ml),
]:
    with open(f'../data/processed/{name}.pkl', 'wb') as f:
        pickle.dump(obj, f)

print("All processed splits saved to data/processed/")

---
**Next:** [Notebook 2 — Model Training & Evaluation](02_model_training_and_evaluation.ipynb)